# Niche-Driven Pseudobulk Differential Expression Analysis

**Aim 2 — Step 4: DE within immune niches across metastasis groups**

Uses immune niche assignments from the Tier 2 NMF analysis to define spot groupings for pseudobulk DE with pyDESeq2.

## Three Approaches

**Approach A — Dominant Niche:** For abundant niches (3, 7, 8, 9, 10, 11), group spots by their dominant immune niche assignment. Ask: within the CAF niche, what genes change between Distant_Mets and No_Mets?

**Approach B — Loading Threshold:** For rare immune niches (1, 4, 5), select spots where the niche loading exceeds the 75th percentile. These spots have meaningful immune cell presence even though the immune niche never "wins" the argmax against stronger stromal signals.

**Approach C — Combined Immune Score:** Sum loadings across all immune-rich niches (1, 4, 5, 7, 8) to define immune-high vs immune-low spots. Compare DE within the same region (CT or Stroma) between immune-high and immune-low microenvironments.

**Requires:** `spot_immune_niche_assignments.csv` and `W_immune_niche_loadings.npy` from `Immune_Niche_Analysis.ipynb`

## Imports

In [ ]:
#!/usr/bin/env python3
"""
Niche-Driven Pseudobulk Differential Expression Analysis
=========================================================
Uses immune niche assignments from Tier 2 NMF to define spot groupings
for pseudobulk DE analysis with pyDESeq2.

Two approaches:
  A) Dominant-niche grouping: For abundant niches (9, 10, 11, etc.),
     group spots by their dominant immune niche assignment.
  B) Loading-threshold grouping: For rare immune niches (1, 4, 5),
     select spots where the niche loading exceeds a threshold,
     capturing immune-enriched microenvironments that never "win" argmax.

Contrasts:
  - Within each niche: Distant_Mets vs No_Mets, LN_Mets vs No_Mets
  - Within each region (CT/Stroma): compare niche-enriched vs niche-depleted spots
  - Immune-high vs immune-low spots across mets groups

Usage: Run cells sequentially in Jupyter.
Requires: spot_immune_niche_assignments.csv from Immune_Niche_Analysis.ipynb
"""

import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import warnings
from datetime import datetime
from scipy import sparse

warnings.filterwarnings('ignore')

## 1. CONFIGURATION

In [ ]:
META_PATH = "/Users/adiallo/Desktop/Thesis/Data_Documents/data_all.csv"
VISIUM_DIR = "/Users/adiallo/Desktop/Thesis/Data_Documents/All_Data/Visium_samples/Visium_Data/samples"
NICHE_DIR = "/Users/adiallo/Desktop/Thesis/Results/AIM_2_Results/Immune_Niche_Analysis"
RESULTS_ROOT = "/Users/adiallo/Desktop/Thesis/Results/AIM_2_Results/Niche_DE_Analysis"
os.makedirs(RESULTS_ROOT, exist_ok=True)

# Niche analysis outputs
NICHE_ASSIGNMENTS_PATH = os.path.join(NICHE_DIR, "spot_immune_niche_assignments.csv")
NICHE_LOADINGS_PATH = os.path.join(NICHE_DIR, "W_immune_niche_loadings.npy")
NICHE_LABELS_PATH = os.path.join(NICHE_DIR, "immune_niche_labels.csv")
NICHE_CATEGORIES_PATH = os.path.join(NICHE_DIR, "immune_niche_category_contributions.csv")

# DE parameters
MIN_SPOTS = 20         # minimum spots per pseudobulk sample
MIN_GENES_EXPR = 10    # gene must be detected in >= this many pseudobulk samples
MIN_SAMPLES_PER_GROUP = 3  # need at least this many samples per contrast group

# Loading threshold for immune niche selection (Approach B)
# Spots in the top X percentile of loading for a given niche
LOADING_PERCENTILE = 75  # use top 25% of spots by loading

# Niche grouping configuration
# Based on K=11 results from immune niche analysis:
#   Niche 1: CD8_T_exhausted + gdT (92.4% immune) — RARE, use loading approach
#   Niche 4: Macrophage + DC (87.9% immune) — moderate frequency
#   Niche 5: B_cell + DC (96.8% immune) — moderate frequency
#   Niche 7: Granulocyte + Macrophage (84.6% immune) — moderate frequency
#   Niche 8: Plasma + DC (89.1% immune) — moderate frequency
#   Niche 3: CAF + Pericyte (67.7% CAF) — CAF-dominant
#   Niche 9: CAF + Fibroblast (40.3% CAF) — abundant
#   Niche 10: CAF + Fibroblast (34.5% CAF) — abundant
#   Niche 11: Pericyte + Fibroblast — dominant (>50% of spots)

# Immune-enriched niches to analyze with loading approach
IMMUNE_NICHES_LOADING = [1, 4, 5]

# Abundant niches to analyze with dominant assignment
ABUNDANT_NICHES_DOMINANT = [3, 7, 8, 9, 10, 11]

# Combined immune score: sum of loadings for immune-rich niches
IMMUNE_SCORE_NICHES = [1, 4, 5, 7, 8]  # all niches with >80% immune content

## 2. HELPER FUNCTIONS

In [ ]:
def classify_mets(row):
    """Classify metastasis status from clinical metadata."""
    any_mets = str(row.get("any_mets", "")).strip().upper() in ["T", "TRUE", "1"]
    dist = str(row.get("Distant_Mets", "")).strip().upper() in ["T", "TRUE", "1"]
    if not any_mets:
        return "No_Mets"
    if dist:
        return "Distant_Mets"
    return "LN_Mets"


def make_volcano(ax, res_df, title_str):
    """Draw a volcano plot."""
    res = res_df.dropna(subset=["padj", "log2FoldChange"])
    if len(res) == 0:
        ax.set_title(f"{title_str}\n(no data)")
        return

    sig_up = (res["padj"] < 0.05) & (res["log2FoldChange"] > 1)
    sig_down = (res["padj"] < 0.05) & (res["log2FoldChange"] < -1)
    ns = ~(sig_up | sig_down)

    ax.scatter(res.loc[ns, "log2FoldChange"], -np.log10(res.loc[ns, "padj"]),
               c="gray", alpha=0.3, s=5)
    ax.scatter(res.loc[sig_up, "log2FoldChange"], -np.log10(res.loc[sig_up, "padj"]),
               c="#D62728", alpha=0.6, s=10, label=f"Up ({sig_up.sum()})")
    ax.scatter(res.loc[sig_down, "log2FoldChange"], -np.log10(res.loc[sig_down, "padj"]),
               c="#1F77B4", alpha=0.6, s=10, label=f"Down ({sig_down.sum()})")

    top_genes = res.head(10).index
    for gene in top_genes:
        if res.loc[gene, "padj"] < 0.05:
            ax.annotate(gene,
                        (res.loc[gene, "log2FoldChange"], -np.log10(res.loc[gene, "padj"])),
                        fontsize=5, alpha=0.8)

    ax.axhline(-np.log10(0.05), color="black", linestyle=":", alpha=0.3)
    ax.axvline(1, color="black", linestyle=":", alpha=0.3)
    ax.axvline(-1, color="black", linestyle=":", alpha=0.3)
    ax.set_xlabel("log2 Fold Change")
    ax.set_ylabel("-log10(padj)")
    ax.set_title(title_str)
    ax.legend(fontsize=8)

## 3. LOAD NICHE ASSIGNMENTS AND METADATA

In [ ]:
print("=" * 70)
print("Niche-Driven Pseudobulk DE Pipeline")
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)

# Load niche assignments
print("\nLoading niche assignments...")
niche_df = pd.read_csv(NICHE_ASSIGNMENTS_PATH)
print(f"  Loaded {len(niche_df)} spot assignments")
print(f"  Columns: {niche_df.columns.tolist()}")

# Load niche loadings (W matrix)
print("Loading niche loadings (W matrix)...")
W = np.load(NICHE_LOADINGS_PATH)
K_FINAL = W.shape[1]
print(f"  W matrix: {W.shape[0]} spots × {K_FINAL} niches")

# Add loadings to niche_df
for k in range(K_FINAL):
    niche_df[f'loading_{k+1}'] = W[:, k]

# Compute combined immune score
immune_loading_cols = [f'loading_{n}' for n in IMMUNE_SCORE_NICHES]
niche_df['immune_score'] = niche_df[immune_loading_cols].sum(axis=1)

# Load niche labels
niche_labels_df = pd.read_csv(NICHE_LABELS_PATH)
niche_labels = dict(zip(niche_labels_df['niche'], niche_labels_df['label']))
print(f"\nNiche labels:")
for k, v in niche_labels.items():
    print(f"  Niche {k}: {v}")

# Load clinical metadata
meta = pd.read_csv(META_PATH)
meta["deident"] = meta["deident"].astype(str)
meta["met_group"] = meta.apply(classify_mets, axis=1)

print(f"\nMets distribution in metadata:")
print(meta["met_group"].value_counts())

## 4. LOAD ALL VISIUM SAMPLES

In [ ]:
print(f"\n{'=' * 70}")
print("Loading Visium samples...")
print("=" * 70)

visium_files = sorted(
    f for f in os.listdir(VISIUM_DIR)
    if f.endswith(".h5ad") and f.replace(".h5ad", "").isdigit()
)
print(f"Found {len(visium_files)} sample files")

sample_cache = {}
for i, f in enumerate(visium_files):
    sid = f.replace(".h5ad", "")
    print(f"  [{i+1}/{len(visium_files)}] {sid}...", end=" ")

    try:
        adata = sc.read_h5ad(os.path.join(VISIUM_DIR, f))

        gene_names = adata.var_names.tolist()
        X = adata.X
        if sparse.issparse(X):
            X = X.toarray()

        # Clinical
        row = meta.loc[meta["deident"] == sid]
        if row.shape[0] != 1:
            print("SKIPPED (metadata mismatch)")
            continue

        clin = row.iloc[0]
        lib_sizes = np.asarray(X.sum(axis=1)).flatten()

        sample_cache[sid] = {
            "X": X,
            "gene_names": gene_names,
            "clin": clin,
            "lib_sizes": lib_sizes,
            "n_spots": X.shape[0],
        }
        print(f"OK ({X.shape[0]} spots, {X.shape[1]} genes)")

    except Exception as e:
        print(f"FAILED: {e}")

print(f"\nCached {len(sample_cache)} samples")

# Shared genes
all_gene_lists = [s["gene_names"] for s in sample_cache.values()]
shared_genes = sorted(set.intersection(*[set(gl) for gl in all_gene_lists]))
print(f"Shared genes: {len(shared_genes)}")

# Pre-compute gene indices
for sid, data in sample_cache.items():
    data["gene_idx"] = [data["gene_names"].index(g) for g in shared_genes]


# ==============================================================================
# 5. BUILD SPOT-LEVEL INDEX MAPPING
#    Map niche_df rows back to per-sample spot indices
# ==============================================================================

## 5. BUILD SPOT-LEVEL INDEX MAPPING

Map niche_df rows back to per-sample spot indices.

In [ ]:
print(f"\n{'=' * 70}")
print("Building spot-to-sample index mapping...")
print("=" * 70)

# The niche_df has all spots concatenated in sample order
# We need to know which rows correspond to which sample
sample_ids_in_niche = niche_df['sample_id'].astype(str).values
unique_samples_niche = []
seen = set()
for s in sample_ids_in_niche:
    if s not in seen:
        unique_samples_niche.append(s)
        seen.add(s)

# Build cumulative index mapping: for each sample, which rows in niche_df
sample_spot_ranges = {}
current = 0
for sid in unique_samples_niche:
    n = (sample_ids_in_niche == sid).sum()
    sample_spot_ranges[sid] = (current, current + n)
    current += n

print(f"  Samples in niche data: {len(sample_spot_ranges)}")
print(f"  Samples in cache: {len(sample_cache)}")
overlap = set(sample_spot_ranges.keys()) & set(sample_cache.keys())
print(f"  Overlap: {len(overlap)}")

## 6. PSEUDOBULK AGGREGATION FUNCTIONS

In [ ]:
def aggregate_pseudobulk_by_mask(sid, spot_mask_within_sample, label=""):
    """
    Given a sample ID and a boolean mask over that sample's spots,
    aggregate raw counts into pseudobulk.
    Returns (counts_array, n_spots, mean_lib_size) or None.
    """
    data = sample_cache[sid]
    X_shared = data["X"][:, data["gene_idx"]]

    if spot_mask_within_sample.sum() < MIN_SPOTS:
        return None

    counts_sum = X_shared[spot_mask_within_sample, :].sum(axis=0)
    counts_sum = np.asarray(counts_sum).flatten()
    n_spots = spot_mask_within_sample.sum()
    mean_lib = np.log1p(np.mean(data["lib_sizes"][spot_mask_within_sample]))

    return counts_sum, n_spots, mean_lib


def run_deseq2(count_df, meta_df, contrast, output_dir, contrast_name):
    """
    Run pyDESeq2 on pseudobulk counts.
    Returns results DataFrame or None.
    """
    from pydeseq2.dds import DeseqDataSet
    from pydeseq2.ds import DeseqStats

    # Check group sizes
    grp_col = contrast[0]
    g1, g2 = contrast[1], contrast[2]
    n1 = (meta_df[grp_col] == g1).sum()
    n2 = (meta_df[grp_col] == g2).sum()

    if n1 < MIN_SAMPLES_PER_GROUP or n2 < MIN_SAMPLES_PER_GROUP:
        print(f"    SKIPPED {contrast_name}: {g1}={n1}, {g2}={n2} (need >= {MIN_SAMPLES_PER_GROUP})")
        return None

    try:
        # Covariates
        meta_df = meta_df.copy()
        meta_df["sex"] = pd.Categorical(meta_df["sex"])
        meta_df["MLH1"] = meta_df["MLH1"].astype(int)

        dds = DeseqDataSet(
            counts=count_df,
            metadata=meta_df,
            design="~ QC + MLH1 + age + sex + group"
        )
        dds.deseq2()

        stat_res = DeseqStats(dds, contrast=contrast, alpha=0.05)
        stat_res.summary()
        res_df = stat_res.results_df.sort_values("padj")

        n_sig = (res_df["padj"] < 0.05).sum()
        n_up = ((res_df["padj"] < 0.05) & (res_df["log2FoldChange"] > 0)).sum()
        n_down = ((res_df["padj"] < 0.05) & (res_df["log2FoldChange"] < 0)).sum()
        print(f"    {contrast_name}: {n_sig} DE genes (↑{n_up} ↓{n_down})")

        res_df.to_csv(os.path.join(output_dir, f"DE_{contrast_name}.csv"))
        return res_df

    except Exception as e:
        print(f"    FAILED {contrast_name}: {e}")
        return None

## 7. APPROACH A — DOMINANT NICHE: DE across mets groups within each niche

In [ ]:
print(f"\n{'=' * 70}")
print("APPROACH A: Pseudobulk DE by dominant niche assignment")
print("=" * 70)

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

approach_a_summary = []

for niche_num in ABUNDANT_NICHES_DOMINANT:
    niche_label = niche_labels.get(niche_num, f"Niche_{niche_num}")
    out_dir = os.path.join(RESULTS_ROOT, f"Approach_A_Niche_{niche_num}_{niche_label.replace(' + ', '_')}")
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n--- Niche {niche_num} ({niche_label}) ---")

    pseudobulk_counts = {}
    pseudobulk_meta = []

    for sid in overlap:
        s, e = sample_spot_ranges[sid]
        niche_spots = niche_df.iloc[s:e]
        region_vals = niche_spots['region'].values
        dominant_niche = niche_spots['immune_niche'].values

        clin = sample_cache[sid]["clin"]

        # Spots assigned to this niche
        niche_mask_global = dominant_niche == niche_num

        # Aggregate per region within this niche
        for region in ["CT", "IM", "Stroma"]:
            region_mask = region_vals == region
            combined_mask = niche_mask_global & region_mask

            result = aggregate_pseudobulk_by_mask(sid, combined_mask, f"{sid}_{region}")
            if result is None:
                continue

            counts_sum, n_spots, mean_lib = result
            sample_key = f"{sid}_{region}"
            pseudobulk_counts[sample_key] = counts_sum

            pseudobulk_meta.append({
                "sample_key": sample_key,
                "patient_id": sid,
                "region": region,
                "mets_status": classify_mets(clin),
                "MLH1": int(clin["MLH1"]),
                "age": int(clin["age"]),
                "sex": clin["sex"],
                "n_spots": n_spots,
                "QC": mean_lib,
            })

    if len(pseudobulk_meta) == 0:
        print(f"  No pseudobulk samples for Niche {niche_num} — skipping")
        continue

    meta_df = pd.DataFrame(pseudobulk_meta).set_index("sample_key")
    count_df = pd.DataFrame(pseudobulk_counts, index=shared_genes).T
    count_df = count_df.loc[meta_df.index].astype(int)

    # Filter low-expressed genes
    genes_detected = (count_df > 0).sum(axis=0)
    keep_genes = genes_detected[genes_detected >= MIN_GENES_EXPR].index
    count_df = count_df[keep_genes]

    print(f"  Pseudobulk samples: {count_df.shape[0]}, Genes: {count_df.shape[1]}")
    print(f"  Region × Mets:")
    print(pd.crosstab(meta_df["region"], meta_df["mets_status"]))

    # Save intermediates
    count_df.to_csv(os.path.join(out_dir, "pseudobulk_counts.csv"))
    meta_df.to_csv(os.path.join(out_dir, "pseudobulk_metadata.csv"))

    # DESeq2 contrasts
    meta_df["group"] = meta_df["region"] + "_" + meta_df["mets_status"]
    meta_df["group"] = pd.Categorical(meta_df["group"])

    contrasts = {
        "Distant_vs_No__CT": ["group", "CT_Distant_Mets", "CT_No_Mets"],
        "Distant_vs_No__Stroma": ["group", "Stroma_Distant_Mets", "Stroma_No_Mets"],
        "LN_vs_No__CT": ["group", "CT_LN_Mets", "CT_No_Mets"],
        "LN_vs_No__Stroma": ["group", "Stroma_LN_Mets", "Stroma_No_Mets"],
        "Stroma_vs_CT__No_Mets": ["group", "Stroma_No_Mets", "CT_No_Mets"],
        "Stroma_vs_CT__Distant_Mets": ["group", "Stroma_Distant_Mets", "CT_Distant_Mets"],
    }

    results_all = {}
    for cname, contrast in contrasts.items():
        res = run_deseq2(count_df, meta_df, contrast, out_dir, cname)
        if res is not None:
            results_all[cname] = res
            n_sig = (res["padj"] < 0.05).sum()
            approach_a_summary.append({
                "niche": niche_num, "label": niche_label,
                "contrast": cname, "n_sig": n_sig,
                "n_up": ((res["padj"] < 0.05) & (res["log2FoldChange"] > 0)).sum(),
                "n_down": ((res["padj"] < 0.05) & (res["log2FoldChange"] < 0)).sum(),
            })

    # Volcano plots
    mets_contrasts = [c for c in results_all if "Distant_vs_No" in c or "LN_vs_No" in c]
    if len(mets_contrasts) > 0:
        ncols = len(mets_contrasts)
        fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 6))
        if ncols == 1:
            axes = [axes]
        fig.suptitle(f"Niche {niche_num} ({niche_label}) — Mets Contrasts",
                     fontsize=13, fontweight="bold")
        for ax, cn in zip(axes, mets_contrasts):
            make_volcano(ax, results_all[cn], cn.replace("__", " | ").replace("_", " "))
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, "volcanos_mets_contrasts.png"), dpi=300, bbox_inches='tight')
        plt.show()

## 8. APPROACH B — LOADING THRESHOLD: Immune-enriched spots

In [ ]:
print(f"\n{'=' * 70}")
print("APPROACH B: Pseudobulk DE using niche loading thresholds")
print("  Selects spots in top percentile of immune niche loading")
print("=" * 70)

approach_b_summary = []

for niche_num in IMMUNE_NICHES_LOADING:
    niche_label = niche_labels.get(niche_num, f"Niche_{niche_num}")
    loading_col = f'loading_{niche_num}'
    out_dir = os.path.join(RESULTS_ROOT, f"Approach_B_Niche_{niche_num}_{niche_label.replace(' + ', '_')}")
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n--- Niche {niche_num} ({niche_label}) — Loading threshold approach ---")

    # Compute threshold from global distribution
    all_loadings = niche_df[loading_col].values
    threshold = np.percentile(all_loadings, LOADING_PERCENTILE)
    print(f"  Loading threshold (p{LOADING_PERCENTILE}): {threshold:.4f}")
    print(f"  Spots above threshold: {(all_loadings > threshold).sum()} "
          f"({(all_loadings > threshold).mean()*100:.1f}%)")

    pseudobulk_counts = {}
    pseudobulk_meta = []

    for sid in overlap:
        s, e = sample_spot_ranges[sid]
        niche_spots = niche_df.iloc[s:e]
        region_vals = niche_spots['region'].values
        loading_vals = niche_spots[loading_col].values

        clin = sample_cache[sid]["clin"]

        # Spots with high loading on this immune niche
        high_loading_mask = loading_vals > threshold

        for region in ["CT", "IM", "Stroma"]:
            region_mask = region_vals == region
            combined_mask = high_loading_mask & region_mask

            result = aggregate_pseudobulk_by_mask(sid, combined_mask)
            if result is None:
                continue

            counts_sum, n_spots, mean_lib = result
            sample_key = f"{sid}_{region}"
            pseudobulk_counts[sample_key] = counts_sum

            pseudobulk_meta.append({
                "sample_key": sample_key,
                "patient_id": sid,
                "region": region,
                "mets_status": classify_mets(clin),
                "MLH1": int(clin["MLH1"]),
                "age": int(clin["age"]),
                "sex": clin["sex"],
                "n_spots": n_spots,
                "QC": mean_lib,
            })

    if len(pseudobulk_meta) == 0:
        print(f"  No pseudobulk samples — skipping")
        continue

    meta_df = pd.DataFrame(pseudobulk_meta).set_index("sample_key")
    count_df = pd.DataFrame(pseudobulk_counts, index=shared_genes).T
    count_df = count_df.loc[meta_df.index].astype(int)

    genes_detected = (count_df > 0).sum(axis=0)
    keep_genes = genes_detected[genes_detected >= MIN_GENES_EXPR].index
    count_df = count_df[keep_genes]

    print(f"  Pseudobulk samples: {count_df.shape[0]}, Genes: {count_df.shape[1]}")
    print(f"  Region × Mets:")
    print(pd.crosstab(meta_df["region"], meta_df["mets_status"]))

    count_df.to_csv(os.path.join(out_dir, "pseudobulk_counts.csv"))
    meta_df.to_csv(os.path.join(out_dir, "pseudobulk_metadata.csv"))

    # DESeq2
    meta_df["group"] = meta_df["region"] + "_" + meta_df["mets_status"]
    meta_df["group"] = pd.Categorical(meta_df["group"])

    contrasts = {
        "Distant_vs_No__CT": ["group", "CT_Distant_Mets", "CT_No_Mets"],
        "Distant_vs_No__Stroma": ["group", "Stroma_Distant_Mets", "Stroma_No_Mets"],
        "LN_vs_No__CT": ["group", "CT_LN_Mets", "CT_No_Mets"],
        "LN_vs_No__Stroma": ["group", "Stroma_LN_Mets", "Stroma_No_Mets"],
    }

    results_all = {}
    for cname, contrast in contrasts.items():
        res = run_deseq2(count_df, meta_df, contrast, out_dir, cname)
        if res is not None:
            results_all[cname] = res
            n_sig = (res["padj"] < 0.05).sum()
            approach_b_summary.append({
                "niche": niche_num, "label": niche_label,
                "contrast": cname, "n_sig": n_sig,
                "n_up": ((res["padj"] < 0.05) & (res["log2FoldChange"] > 0)).sum(),
                "n_down": ((res["padj"] < 0.05) & (res["log2FoldChange"] < 0)).sum(),
            })

    # Volcanos
    if len(results_all) > 0:
        ncols = len(results_all)
        fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 6))
        if ncols == 1:
            axes = [axes]
        fig.suptitle(f"Niche {niche_num} ({niche_label}) — Loading Threshold DE",
                     fontsize=13, fontweight="bold")
        for ax, (cn, res) in zip(axes, results_all.items()):
            make_volcano(ax, res, cn.replace("__", " | ").replace("_", " "))
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, "volcanos_loading_threshold.png"), dpi=300, bbox_inches='tight')
        plt.show()

## 9. APPROACH C — COMBINED IMMUNE SCORE: High vs Low immune spots

In [ ]:
print(f"\n{'=' * 70}")
print("APPROACH C: Immune-high vs Immune-low spots")
print(f"  Combined immune score from niches: {IMMUNE_SCORE_NICHES}")
print("=" * 70)

out_dir_c = os.path.join(RESULTS_ROOT, "Approach_C_ImmuneScore")
os.makedirs(out_dir_c, exist_ok=True)

# Define high/low by median split per sample
immune_scores = niche_df['immune_score'].values
score_median = np.median(immune_scores)
score_q75 = np.percentile(immune_scores, 75)

print(f"  Global immune score: median={score_median:.4f}, Q75={score_q75:.4f}")

for threshold_name, threshold_val in [("median", score_median), ("Q75", score_q75)]:
    print(f"\n  --- Threshold: {threshold_name} ({threshold_val:.4f}) ---")

    pseudobulk_counts = {}
    pseudobulk_meta = []

    for sid in overlap:
        s, e = sample_spot_ranges[sid]
        niche_spots = niche_df.iloc[s:e]
        region_vals = niche_spots['region'].values
        immune_scores_sample = niche_spots['immune_score'].values

        clin = sample_cache[sid]["clin"]

        high_mask = immune_scores_sample >= threshold_val
        low_mask = immune_scores_sample < threshold_val

        for region in ["CT", "Stroma"]:
            region_mask = region_vals == region

            for level, level_mask in [("ImmuneHigh", high_mask), ("ImmuneLow", low_mask)]:
                combined = region_mask & level_mask
                result = aggregate_pseudobulk_by_mask(sid, combined)
                if result is None:
                    continue

                counts_sum, n_spots, mean_lib = result
                sample_key = f"{sid}_{region}_{level}"
                pseudobulk_counts[sample_key] = counts_sum

                pseudobulk_meta.append({
                    "sample_key": sample_key,
                    "patient_id": sid,
                    "region": region,
                    "immune_level": level,
                    "mets_status": classify_mets(clin),
                    "MLH1": int(clin["MLH1"]),
                    "age": int(clin["age"]),
                    "sex": clin["sex"],
                    "n_spots": n_spots,
                    "QC": mean_lib,
                })

    if len(pseudobulk_meta) == 0:
        print(f"  No pseudobulk samples — skipping")
        continue

    meta_df = pd.DataFrame(pseudobulk_meta).set_index("sample_key")
    count_df = pd.DataFrame(pseudobulk_counts, index=shared_genes).T
    count_df = count_df.loc[meta_df.index].astype(int)

    genes_detected = (count_df > 0).sum(axis=0)
    keep_genes = genes_detected[genes_detected >= MIN_GENES_EXPR].index
    count_df = count_df[keep_genes]

    print(f"  Samples: {count_df.shape[0]}, Genes: {count_df.shape[1]}")

    sub_out = os.path.join(out_dir_c, f"threshold_{threshold_name}")
    os.makedirs(sub_out, exist_ok=True)
    count_df.to_csv(os.path.join(sub_out, "pseudobulk_counts.csv"))
    meta_df.to_csv(os.path.join(sub_out, "pseudobulk_metadata.csv"))

    # Group: region × immune_level × mets
    meta_df["group"] = meta_df["region"] + "_" + meta_df["immune_level"] + "_" + meta_df["mets_status"]
    meta_df["group"] = pd.Categorical(meta_df["group"])

    contrasts = {
        # Immune high vs low within CT, same mets status
        "ImmuneHigh_vs_Low__CT_NoMets": ["group", "CT_ImmuneHigh_No_Mets", "CT_ImmuneLow_No_Mets"],
        "ImmuneHigh_vs_Low__CT_DistantMets": ["group", "CT_ImmuneHigh_Distant_Mets", "CT_ImmuneLow_Distant_Mets"],
        # Mets comparison within immune-high CT spots
        "Distant_vs_No__CT_ImmuneHigh": ["group", "CT_ImmuneHigh_Distant_Mets", "CT_ImmuneHigh_No_Mets"],
        # Mets comparison within immune-high Stroma spots
        "Distant_vs_No__Stroma_ImmuneHigh": ["group", "Stroma_ImmuneHigh_Distant_Mets", "Stroma_ImmuneHigh_No_Mets"],
        # Immune high vs low within Stroma
        "ImmuneHigh_vs_Low__Stroma_NoMets": ["group", "Stroma_ImmuneHigh_No_Mets", "Stroma_ImmuneLow_No_Mets"],
    }

    results_all = {}
    for cname, contrast in contrasts.items():
        res = run_deseq2(count_df, meta_df, contrast, sub_out, cname)
        if res is not None:
            results_all[cname] = res

    if len(results_all) > 0:
        ncols = min(3, len(results_all))
        nrows = int(np.ceil(len(results_all) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))
        axes_flat = np.array(axes).flatten() if len(results_all) > 1 else [axes]
        fig.suptitle(f"Immune Score DE — {threshold_name} threshold",
                     fontsize=13, fontweight="bold")
        for idx, (cn, res) in enumerate(results_all.items()):
            if idx < len(axes_flat):
                make_volcano(axes_flat[idx], res, cn.replace("__", "\n").replace("_", " "))
        for idx in range(len(results_all), len(axes_flat)):
            axes_flat[idx].axis('off')
        plt.tight_layout()
        plt.savefig(os.path.join(sub_out, "volcanos_immune_score.png"), dpi=300, bbox_inches='tight')
        plt.show()

## 10. SUMMARY

In [ ]:
print(f"\n{'=' * 70}")
print("SUMMARY")
print("=" * 70)

if approach_a_summary:
    print("\nApproach A — Dominant Niche DE:")
    summary_a = pd.DataFrame(approach_a_summary)
    print(summary_a.to_string(index=False))
    summary_a.to_csv(os.path.join(RESULTS_ROOT, "summary_approach_A.csv"), index=False)

if approach_b_summary:
    print("\nApproach B — Loading Threshold DE:")
    summary_b = pd.DataFrame(approach_b_summary)
    print(summary_b.to_string(index=False))
    summary_b.to_csv(os.path.join(RESULTS_ROOT, "summary_approach_B.csv"), index=False)

print(f"\nAll results saved to: {RESULTS_ROOT}")
print(f"Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")